# Notebook 03: Tiling Pipeline (tile_and_mask.py)

**Module:** 12 Capstone — water-bodies-detection  
**Duration:** ~3 hrs

---

## Learning Objectives

1. Understand windowed GeoTIFF reading
2. Trace band selection and robust normalization
3. Follow dual mask rasterization
4. Explain negative tile sampling

## tile_and_mask.py Purpose

Converts raw **GeoTIFF + aquaculture shapefile** into ML-ready triplets:

```
output_dir/
├── tiles/           # 6-band float32 normalized
├── masks_aqua/      # binary interior
├── masks_boundary/  # dilated bund lines
└── tiling_meta.json # reproducibility
```

In [ ]:
import os, json, yaml
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (8, 5)
REPO = '../../water-bodies-detection'
print('Capstone repo path:', os.path.abspath(REPO) if os.path.exists(REPO) else 'clone water-bodies-detection alongside Machine-Learning')

## Key Functions

### `robust_normalize_tile(tile, p_low, p_high)`
- Per-band percentile scaling (2nd–98th)
- Fallback: minmax → sigma → flat
- NaN outside valid pixels
- **Why:** Planet scenes vary in radiometry

### `pixel_valid_mask`
- Alpha band (index 9) ≥ min_alpha_value
- Exclude all-bands-zero padding
- Exclude nodata

### Mask generation
1. **Aqua:** `rasterio.features.rasterize` polygon fills
2. **Boundary:** rasterize `polygon.boundary`, dilate by `boundary_width_meters / (2 × GSD)`

### Negative tiles
- Random windows with both masks = 0
- `negative_tile_ratio: 0.2` — teaches background class
- Optional `--hard_negative_shp` for rivers/lakes

In [ ]:
# Normalization concept (from tile_and_mask.py)
def demo_percentile_norm(band, p_low=2, p_high=98):
    lo, hi = np.percentile(band, [p_low, p_high])
    return np.clip((band - lo) / (hi - lo + 1e-8), 0, 1)

band = np.random.exponential(500, (512, 512)) + 100
norm = demo_percentile_norm(band)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].imshow(band, cmap='gray'); ax[0].set_title('Raw DN'); ax[0].axis('off')
ax[1].imshow(norm, cmap='gray'); ax[1].set_title('Percentile normalized'); ax[1].axis('off')
plt.tight_layout(); plt.show()

## CLI

```bash
python tile_and_mask.py \
  --input_tif area.tif \
  --input_shp aquaculture.shp \
  --output_dir tiles_masks/ \
  --config config/default.yaml
```

## Common Bugs
- CRS mismatch between SHP and TIF → reproject in script
- `meters_per_pixel` null → boundary width wrong
- Too few positive tiles → check `min_positive_valid_pixels`

---

## Summary

tile_and_mask.py is the data foundation — garbage in, garbage out.

**Next:** [04_GIS_Fundamentals.ipynb](04_GIS_Fundamentals.ipynb)